In [6]:
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch
import pandas as pd

# 1. Load and prepare the dataset using pandas
df = pd.read_csv("data.csv", encoding="ISO-8859-1")

# Convert pandas DataFrame to datasets.Dataset format
dataset = Dataset.from_pandas(df)

# Split the dataset into train and validation sets (optional)
train_dataset = dataset

# 2. Load the T5 model and tokenizer
model_name = "t5-small"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Move model to GPU if available
model.to(device)

# 3. Preprocess the dataset
def preprocess_function(examples):
    inputs = [f"question: {q} context: {c}" for q, c in zip(examples['question'], examples['context'])]
    targets = examples['answer']  # The answer is the target/output
    
    # Ensure all inputs and targets are strings
    inputs = [str(input_example) for input_example in inputs]
    targets = [str(target_example) for target_example in targets]
    
    # Tokenize inputs and targets
    model_inputs = tokenizer(inputs, max_length=512, padding=True, truncation=True)
    labels = tokenizer(targets, max_length=128, padding=True, truncation=True)
    
    # Add labels to model inputs
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

# Apply preprocessing
tokenized_dataset = train_dataset.map(preprocess_function, batched=True, keep_in_memory=True)

# 4. Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=2,  # Reduce batch size for 4GB VRAM
    gradient_accumulation_steps=2,  # Simulate larger batch size
    save_steps=10_000,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=500,
    evaluation_strategy="no",  # Disable evaluation
    fp16=True,  # Enable mixed precision
)

# 5. Initialize the Trainer
trainer = Trainer(
    model=model,                         # the model to be trained
    args=training_args,                  # training arguments
    train_dataset=tokenized_dataset,     # training dataset
    tokenizer=tokenizer,                 # tokenizer for text processing
)

# 6. Train the model
trainer.train()

# 7. Save the fine-tuned model
model.save_pretrained('./fine_tuned_t5')
tokenizer.save_pretrained('./fine_tuned_t5')

# 8. Optionally, evaluate the model
eval_dataset = tokenized_dataset.select(range(len(tokenized_dataset) // 2))  # Example split
results = trainer.evaluate(eval_dataset=eval_dataset)

# Print evaluation results
print(results)

Using device: cuda


  0%|          | 0/1 [00:00<?, ?ba/s]

C:\Users\kavin\AppData\Roaming\Python\Python312\site-packages\transformers\training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\kavin\AppData\Local\Temp\ipykernel_300456\2812574704.py:63: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


  0%|          | 0/6 [00:00<?, ?it/s]

{'train_runtime': 1.8944, 'train_samples_per_second': 14.252, 'train_steps_per_second': 3.167, 'train_loss': 18.143702189127605, 'epoch': 2.0}


  0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 16.651180267333984, 'eval_runtime': 0.0433, 'eval_samples_per_second': 92.311, 'eval_steps_per_second': 23.078, 'epoch': 2.0}


In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

# Load the fine-tuned model and tokenizer
model_path = './fine_tuned_t5'
model = T5ForConditionalGeneration.from_pretrained(model_path)
tokenizer = T5Tokenizer.from_pretrained(model_path)

# Check if a GPU is available and move model to GPU if so
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Define the question and context
question = "what supervised learninng?"
context = "Supervised learning is a type of machine learning where the model is trained using labeled data. The goal is to teach the model to make predictions based on known outcomes."
# Format input
input_text = f"question: {question} context: {context}"

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).input_ids

# Move input tensors to the same device as the model (GPU or CPU)
input_ids = input_ids.to(device)

# Generate the output
output_ids = model.generate(input_ids, max_length=250, num_beams=5, early_stopping=True)

# Decode the output to text
answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Answer: {answer}")

Answer: machine learning
